## Curso: Análise e Desenvolvimento de Sistemas

## Disciplina: DISRUPTIVE ARCHITECTURES: IOT, IA & GENERATIVE AI

## Turmas: 2TDS (1º Semestre de 2026)

## Professor: André Tritiack

## Autores:
- **Ranaldo José da Silva** — RM 559210
- **Otoniel Arantes Barbado** — RM 560112

---

## **Diagnóstico médico: Diabetes no Povo Pima**

Este dataset contém dados sobre incidência de diabetes em mulheres do **Povo Pima** (índios nativos norte-americanos originários do atual Estado do Arizona).

Contém **8 atributos biomédicos** para **768 entradas anonimizadas**:
- **500** pacientes testados **negativo** para diabetes (Outcome = 0)
- **268** pacientes testados **positivo** para diabetes (Outcome = 1)

### Atributos do dataset:
| Atributo | Descrição |
|----------|-----------|
| Pregnancies | Número de gestações |
| Glucose | Concentração de glicose no plasma |
| BloodPressure | Pressão arterial diastólica (mm Hg) |
| SkinThickness | Espessura da dobra cutânea do tríceps (mm) |
| Insulin | Insulina sérica em 2 horas (mu U/ml) |
| BMI | Índice de massa corporal (peso/altura²) |
| DiabetesPedigreeFunction | Função de pedigree de diabetes (histórico familiar) |
| Age | Idade (anos) |
| Outcome | Resultado: 0 = Negativo, 1 = Positivo |

Fonte: https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database

### Algoritmos utilizados:
- **Algoritmo 1:** KNN (K-Nearest Neighbors)
- **Algoritmo 2:** Random Forest

---
## 1. Importação das Bibliotecas

In [4]:
# Importação das bibliotecas necessárias para o projeto

import pandas as pd       # Manipulação e análise de dados em formato tabular (DataFrames)
import numpy as np        # Operações matemáticas e manipulação de arrays numéricos

# Divisão dos dados em conjuntos de treino e teste
from sklearn.model_selection import train_test_split

# Algoritmo KNN: classifica com base nos K vizinhos mais próximos
from sklearn.neighbors import KNeighborsClassifier

# Algoritmo Random Forest: conjunto de árvores de decisão para maior precisão
from sklearn.ensemble import RandomForestClassifier

# Métrica para avaliar a performance dos modelos
from sklearn.metrics import accuracy_score

# Biblioteca para salvar e carregar o modelo treinado em arquivo
import pickle

---
## 2. Carregamento e Exploração dos Dados

In [5]:
# Criação do DataFrame a partir do arquivo CSV do dataset Pima Indians Diabetes
# O arquivo diabetes.csv deve estar na mesma pasta deste notebook
df = pd.read_csv('diabetes.csv')

In [6]:
# Visualiza o tamanho do dataframe: (linhas, colunas)
# Esperado: (768, 9) — 768 pacientes e 9 colunas (8 atributos + 1 alvo)
df.shape

(768, 9)

In [7]:
# Visualiza os 5 primeiros registros do dataset
# Permite ter uma visão geral dos dados e verificar se foram carregados corretamente
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [8]:
# Exibe informações técnicas sobre o dataframe:
# - Nome e tipo de cada coluna (int64, float64)
# - Quantidade de valores não-nulos por coluna
# - Uso de memória
# Importante para identificar colunas com valores ausentes ou tipos incorretos
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [9]:
# Exibe estatísticas descritivas de cada atributo numérico:
# count, mean (média), std (desvio padrão), min, max e percentis (25%, 50%, 75%)
# Útil para entender a distribuição dos dados e identificar possíveis outliers
df.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In [10]:
# Verifica a distribuição da coluna alvo (Outcome)
# 0 = Negativo para diabetes
# 1 = Positivo para diabetes
# Importante para verificar se o dataset está balanceado
df['Outcome'].value_counts()

Outcome
0    500
1    268
Name: count, dtype: int64

---
## 3. Separação das Variáveis X e y

- **X**: matriz de features (atributos de entrada) — os 8 atributos biomédicos
- **y**: vetor alvo (variável que queremos prever) — coluna `Outcome`

In [11]:
# Criação da variável X com todas as colunas EXCETO a coluna alvo 'Outcome'
# X contém os 8 atributos biomédicos que serão usados para fazer a previsão
X = df.drop('Outcome', axis=1)

In [12]:
# Criação da variável y com apenas a coluna alvo 'Outcome'
# y contém os rótulos que o modelo deverá aprender a prever (0 ou 1)
# .values converte o Series pandas para um array NumPy
y = df['Outcome'].values

In [13]:
# Visualiza os primeiros registros de X para confirmar que os atributos estão corretos
# Não deve conter a coluna 'Outcome'
X.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,6,148,72,35,0,33.6,0.627,50
1,1,85,66,29,0,26.6,0.351,31
2,8,183,64,0,0,23.3,0.672,32
3,1,89,66,23,94,28.1,0.167,21
4,0,137,40,35,168,43.1,2.288,33


In [14]:
# Visualiza X convertido para array NumPy
# O modelo de Machine Learning trabalha internamente com arrays NumPy
X.values

array([[  6.   , 148.   ,  72.   , ...,  33.6  ,   0.627,  50.   ],
       [  1.   ,  85.   ,  66.   , ...,  26.6  ,   0.351,  31.   ],
       [  8.   , 183.   ,  64.   , ...,  23.3  ,   0.672,  32.   ],
       ...,
       [  5.   , 121.   ,  72.   , ...,  26.2  ,   0.245,  30.   ],
       [  1.   , 126.   ,  60.   , ...,  30.1  ,   0.349,  47.   ],
       [  1.   ,  93.   ,  70.   , ...,  30.4  ,   0.315,  23.   ]],
      shape=(768, 8))

In [15]:
# Visualiza y como array NumPy
# Deve conter apenas valores 0 (negativo) e 1 (positivo)
y

array([1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0,
       1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1,
       0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0,
       1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0,
       1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1,
       1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1,
       1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
       1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1,
       0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1,
       1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1,
       1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0,
       1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0,
       1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0,
       0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1,

---
## 4. Divisão dos Dados em Treino e Teste

Separamos **70%** dos dados para **treino** (o modelo aprende com eles) e **30%** para **teste** (avaliamos a performance em dados que o modelo nunca viu).

In [16]:
# Divisão dos dados em conjuntos de treino (70%) e teste (30%)
# test_size=0.3  → 30% dos dados para teste
# random_state=42 → semente aleatória para garantir reprodutibilidade
#                   (mesmos resultados toda vez que o código for executado)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Exibe o tamanho de cada conjunto
print(f'Treino: {X_train.shape[0]} amostras')
print(f'Teste:  {X_test.shape[0]} amostras')

Treino: 537 amostras
Teste:  231 amostras


---
## 5. Modelo 1 — KNN (K-Nearest Neighbors)

O KNN classifica um novo ponto com base nos **K vizinhos mais próximos** no espaço dos atributos. É um algoritmo simples e intuitivo, mas pode ser menos preciso em dados com muitas dimensões.

In [17]:
# Criação da instância do primeiro modelo: KNN
# n_neighbors=5 → o modelo considera os 5 vizinhos mais próximos para classificar
# A classe majoritária entre os 5 vizinhos define a previsão final
modelo_knn = KNeighborsClassifier(n_neighbors=5)

In [18]:
# Treinamento do modelo KNN com os dados de treino
# O modelo aprende os padrões dos dados de entrada (X_train)
# associando-os aos rótulos corretos (y_train)
modelo_knn.fit(X_train, y_train)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.Doesn't affect :meth:`fit` method.",None


In [19]:
# Geração das previsões do KNN para os dados de teste
# O modelo usa o que aprendeu no treino para prever os dados que nunca viu (X_test)
y_predict_1 = modelo_knn.predict(X_test)

In [20]:
# Visualiza o array de previsões do KNN
# Cada valor é 0 (negativo) ou 1 (positivo) para cada paciente do conjunto de teste
y_predict_1

array([1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0,
       1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1,
       0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0,
       0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0,
       0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1,
       0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0,
       0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0,
       0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0,
       0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0,
       0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1,
       1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0])

---
## 6. Modelo 2 — Random Forest

O Random Forest cria **múltiplas árvores de decisão** com subconjuntos aleatórios dos dados e combina os resultados (votação majoritária). Geralmente é mais preciso e robusto que uma única árvore ou o KNN.

In [21]:
# Criação da instância do segundo modelo: Random Forest
# n_estimators=100 → cria 100 árvores de decisão diferentes
# random_state=42  → garante reprodutibilidade dos resultados
# A previsão final é a classe mais votada entre as 100 árvores
modelo_rf = RandomForestClassifier(n_estimators=100, random_state=42)

In [22]:
# Treinamento do modelo Random Forest com os dados de treino
# Cada uma das 100 árvores é treinada com uma amostra aleatória dos dados
# Isso reduz o overfitting e melhora a generalização do modelo
modelo_rf.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [23]:
# Geração das previsões do Random Forest para os dados de teste
# As 100 árvores votam e a classe majoritária é a previsão final
y_predict_2 = modelo_rf.predict(X_test)

In [24]:
# Visualiza o array de previsões do Random Forest
# Cada valor é 0 (negativo) ou 1 (positivo) para cada paciente do conjunto de teste
y_predict_2

array([0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0,
       0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0,
       0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1,
       0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1,
       0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1,
       0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1,
       0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0,
       0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0,
       0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0,
       0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1,
       1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0])

---
## 7. Avaliação dos Modelos

A **acurácia** mede a proporção de previsões corretas em relação ao total de previsões. Quanto mais próximo de 1.0 (100%), melhor o modelo.

In [25]:
# Cálculo da acurácia do Modelo 1 (KNN)
# Compara as previsões (y_predict_1) com os valores reais (y_test)
# e calcula a proporção de acertos
acc_knn = accuracy_score(y_test, y_predict_1)
print(f'Acurácia do KNN: {acc_knn:.4f} ({acc_knn*100:.2f}%)')

Acurácia do KNN: 0.6883 (68.83%)


In [26]:
# Cálculo da acurácia do Modelo 2 (Random Forest)
# Compara as previsões (y_predict_2) com os valores reais (y_test)
acc_rf = accuracy_score(y_test, y_predict_2)
print(f'Acurácia do Random Forest: {acc_rf:.4f} ({acc_rf*100:.2f}%)')

Acurácia do Random Forest: 0.7532 (75.32%)


In [27]:
# Comparação entre os dois modelos
# Indica qual modelo deve ser salvo para uso na API
print('\n--- Comparação dos Modelos ---')
print(f'KNN           : {acc_knn*100:.2f}%')
print(f'Random Forest : {acc_rf*100:.2f}%')
print(f'\nMelhor modelo : {"Random Forest" if acc_rf >= acc_knn else "KNN"}')


--- Comparação dos Modelos ---
KNN           : 68.83%
Random Forest : 75.32%

Melhor modelo : Random Forest


---
## 8. Salvando o Modelo

O modelo treinado é salvo em um arquivo `.pkl` (formato pickle) para ser reutilizado na API Flask sem precisar treinar novamente.

In [28]:
import pickle

# O Random Forest obteve maior acurácia (~75.32%) versus KNN (~68.83%)
# Por isso, salvamos o Random Forest como o modelo oficial da API
modelo = modelo_rf

# Salva o modelo em arquivo binário .pkl usando pickle
# 'wb' = write binary (escrita em modo binário)
# O arquivo modelo_diabetes.pkl será usado pela API Flask para fazer previsões
with open('modelo_diabetes.pkl', 'wb') as f:
    pickle.dump(modelo, f)

print('✅ Modelo salvo com sucesso como modelo_diabetes.pkl')
print(f'   Algoritmo: Random Forest')
print(f'   Acurácia:  {acc_rf*100:.2f}%')

✅ Modelo salvo com sucesso como modelo_diabetes.pkl
   Algoritmo: Random Forest
   Acurácia:  75.32%
